# Task 3 - Athena Queries And Dashboard

This notebook serves the curated star schema produced by the Glue ETL in this folder. It registers the Parquet outputs in Glue/Athena, runs the three required analytical queries, and builds an interactive dashboard over the detailed sales dataset.

## Setup

Before running the notebook, run the Terraform pipeline and the Glue job. Then replace `S3_BUCKET_NAME` with the value from `terraform output s3_bucket_name`. The curated path is produced by `terraform/main.tf` as `s3://<bucket>/curated/`.

In [13]:
import json
import os
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def load_env_file(env_path):
    if not env_path.exists():
        return
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in raw_line:
            continue
        key, value = raw_line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


load_env_file(Path(".env"))
load_env_file(Path("assignment_1/task_3/grupo_5/antonio/.env"))

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
GLUE_DATABASE = "classicmodels_analytics_grupo5_antonio"

outputs_candidates = [
    Path("terraform/outputs.json"),
    Path("assignment_1/task_3/grupo_5/antonio/terraform/outputs.json"),
]
outputs_path = next((candidate for candidate in outputs_candidates if candidate.exists()), None)
if outputs_path is not None:
    terraform_outputs = json.loads(outputs_path.read_text())
    S3_BUCKET_NAME = terraform_outputs["s3_bucket_name"]["value"]
else:
    S3_BUCKET_NAME = "replace-with-terraform-output-bucket"

BASE_PATH = f"s3://{S3_BUCKET_NAME}/curated"
ATHENA_OUTPUT = f"s3://{S3_BUCKET_NAME}/athena-results/"

required_aws_env = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]
missing_aws_env = [name for name in required_aws_env if not os.environ.get(name)]
if missing_aws_env:
    raise RuntimeError(
        "Missing AWS credentials in the notebook environment: "
        + ", ".join(missing_aws_env)
        + ". Run Jupyter from assignment_1/task_3/grupo_5/antonio or keep the .env file there."
    )

boto3_session = boto3.Session(region_name=AWS_REGION)
boto3.setup_default_session(region_name=AWS_REGION)
wr.config.athena_query_wait_polling_delay = 1

print(f"Using bucket: {S3_BUCKET_NAME}")
print(f"Using Glue database: {GLUE_DATABASE}")


Using bucket: grupo5-task3-61f89c39
Using Glue database: classicmodels_analytics_grupo5_antonio


## Catalog The Curated Parquet Outputs

The Glue ETL writes one Parquet folder per star-schema table. This cell creates the Glue/Athena database if needed and registers the five required tables.

In [14]:
expected_tables = [
    "fact_orders",
    "dim_customers",
    "dim_products",
    "dim_dates",
    "dim_countries",
]

existing_databases = set(wr.catalog.databases(boto3_session=boto3_session)["Database"].tolist())
if GLUE_DATABASE not in existing_databases:
    wr.catalog.create_database(GLUE_DATABASE, boto3_session=boto3_session)

for table in expected_tables:
    path = f"{BASE_PATH}/{table}/"
    objects = wr.s3.list_objects(path, boto3_session=boto3_session)
    if not objects:
        raise FileNotFoundError(f"No Parquet objects found for {table}: {path}")

    wr.s3.store_parquet_metadata(
        path=path,
        database=GLUE_DATABASE,
        table=table,
        dataset=True,
        mode="overwrite",
        boto3_session=boto3_session,
    )

print(f"Cataloged {len(expected_tables)} tables in {GLUE_DATABASE}.")


Cataloged 5 tables in classicmodels_analytics_grupo5_antonio.


## Query 1 - Product Dimension Exploration

In [15]:
products_query = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = wr.athena.read_sql_query(
    sql=products_query,
    database=GLUE_DATABASE,
    s3_output=ATHENA_OUTPUT,
    boto3_session=boto3_session,
)

display(df_products.head(20))


,product_id,product_name,product_line,product_vendor
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,Min Lin Diecast
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,Classic Metal Creations
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,Highway 66 Mini Classics
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,Red Start Diecast
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,Motor City Art Classics
5,S10_4962,1962 LanciaA Delta 16V,Classic Cars,Second Gear Diecast
6,S12_1099,1968 Ford Mustang,Classic Cars,Autoart Studio Design
7,S12_4675,1969 Dodge Charger,Classic Cars,Welly Diecast Productions
8,S18_1097,1940 Ford Pickup Truck,Trucks and Buses,Studio M Art Models
9,S18_1129,1993 Mazda RX-7,Classic Cars,Highway 66 Mini Classics


## Query 2 - Total Sales By Country

In [16]:
country_sales_query = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_country_sales = wr.athena.read_sql_query(
    sql=country_sales_query,
    database=GLUE_DATABASE,
    s3_output=ATHENA_OUTPUT,
    boto3_session=boto3_session,
)

display(df_country_sales)


,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


## Query 3 - Detailed Analytical Dataset

In [17]:
detail_query = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_analytics = wr.athena.read_sql_query(
    sql=detail_query,
    database=GLUE_DATABASE,
    s3_output=ATHENA_OUTPUT,
    boto3_session=boto3_session,
)

df_analytics["full_date"] = pd.to_datetime(df_analytics["full_date"])
df_analytics["total_sales"] = pd.to_numeric(df_analytics["total_sales"])
df_analytics["country"] = df_analytics["country"].astype(str).str.strip()

print(f"Loaded {len(df_analytics)} analytical rows.")
display(df_analytics.head())


Loaded 2996 analytical rows.


,full_date,product_line,product_name,country,total_sales
0,2003-01-29,Trucks and Buses,1958 Setra Bus,Norway,3284.28
1,2003-01-29,Trucks and Buses,1940 Ford Pickup Truck,Norway,3307.50
2,2003-01-29,Vintage Cars,1934 Ford V8 Coupe,Norway,2164.40
3,2003-01-29,Vintage Cars,18th Century Vintage Horse Carriage,Norway,2173.00
4,2003-01-29,Vintage Cars,1939 Cadillac Limousine,Norway,1670.75


## Interactive Dashboard

In [18]:
country_options = ["All"] + sorted(df_analytics["country"].dropna().unique().tolist())
product_line_options = ["All"] + sorted(df_analytics["product_line"].dropna().unique().tolist())

start_date_widget = widgets.DatePicker(
    description="Start date",
    value=df_analytics["full_date"].min().date(),
)
end_date_widget = widgets.DatePicker(
    description="End date",
    value=df_analytics["full_date"].max().date(),
)
country_widget = widgets.Dropdown(
    options=country_options,
    value="All",
    description="Country",
)
product_line_widget = widgets.Dropdown(
    options=product_line_options,
    value="All",
    description="Product line",
    style={"description_width": "initial"},
)
top_n_widget = widgets.IntSlider(
    min=1,
    max=10,
    value=5,
    description="Top N",
)


def update_dashboard(start_date, end_date, country, product_line, top_n):
    filtered = df_analytics.copy()

    if start_date is not None:
        filtered = filtered[filtered["full_date"].dt.date >= start_date]
    if end_date is not None:
        filtered = filtered[filtered["full_date"].dt.date <= end_date]
    if country != "All":
        filtered = filtered[filtered["country"] == country]
    if product_line != "All":
        filtered = filtered[filtered["product_line"] == product_line]

    ranked = (
        filtered.groupby("product_name", as_index=False)["total_sales"]
        .sum()
        .sort_values("total_sales", ascending=False)
        .head(top_n)
    )

    if ranked.empty:
        print("No rows match the selected filters.")
        return

    plt.figure(figsize=(10, 6))
    sns.barplot(data=ranked, x="total_sales", y="product_name", color="#2f6f9f")
    plt.title(f"Top {top_n} products by total sales")
    plt.xlabel("Total sales")
    plt.ylabel("Product")
    plt.grid(axis="x", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()


controls = widgets.VBox([
    widgets.HBox([start_date_widget, end_date_widget]),
    widgets.HBox([country_widget, product_line_widget, top_n_widget]),
])

output = widgets.interactive_output(
    update_dashboard,
    {
        "start_date": start_date_widget,
        "end_date": end_date_widget,
        "country": country_widget,
        "product_line": product_line_widget,
        "top_n": top_n_widget,
    },
)

display(controls, output)


Output()